In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from xgboost import XGBClassifier
import shap

sns.set_theme(style='darkgrid')
print("All imports OK")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df = pd.read_csv('../data/processed/transcripts_with_prices.csv')
df['date'] = pd.to_datetime(df['date'])

# ── Text features ──────────────────────────────────────────
df['transcript_length']  = df['transcript'].str.split().str.len()
df['avg_word_length']    = df['transcript'].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if pd.notna(x) else 0)
df['question_count']     = df['transcript'].str.count(r'\?')
df['exclaim_count']      = df['transcript'].str.count(r'\!')

# ── Signal word counts ──────────────────────────────────────
signal_words = ['growth','revenue','guidance','margin','beat','miss',
                'raised','lowered','uncertainty','headwinds','record',
                'strong','weak','declined','exceeded','profit','loss',
                'outlook','forecast','demand']

for word in signal_words:
    df[f'word_{word}'] = df['transcript'].str.lower().str.count(word)

# ── Time features ───────────────────────────────────────────
df['quarter_num'] = df['q'].str.extract(r'Q(\d)').astype(float)
df['year']        = df['date'].dt.year

# ── Drop nulls ──────────────────────────────────────────────
feature_cols = (['transcript_length','avg_word_length',
                 'question_count','exclaim_count',
                 'quarter_num','year']
                + [f'word_{w}' for w in signal_words])

df_ml = df[feature_cols + ['direction']].dropna()
print(f"ML dataset shape: {df_ml.shape}")
print(f"Feature count: {len(feature_cols)}")
print(f"Class balance — Up: {df_ml['direction'].sum()} | Down: {(df_ml['direction']==0).sum()}")

In [ ]:
X = df_ml[feature_cols]
y = df_ml['direction']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

print("✅ Model trained!")
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

In [ ]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("=" * 45)
print("CLASSIFICATION REPORT")
print("=" * 45)
print(classification_report(y_test, y_pred,
      target_names=['Down', 'Up']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f}")

# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
print(f"\n5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Down','Up'],
            yticklabels=['Down','Up'], ax=axes[0])
axes[0].set_title('Confusion matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'AUC = {roc_auc_score(y_test, y_pred_prob):.3f}')
axes[1].plot([0,1],[0,1], 'r--', linewidth=1)
axes[1].set_title('ROC curve')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/ml_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test,
                  feature_names=feature_cols,
                  plot_type='bar', show=False)
plt.title('Feature importance (SHAP values)')
plt.tight_layout()
plt.savefig('../data/processed/ml_shap.png', bbox_inches='tight')
plt.show()